# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts

This notebook simulates Gold metric calculations (like readmissions) and role-based column masking using DuckDB and Pandas.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')
print("DuckDB in-memory connection established.")

## Simulate dbt Gold Mart (Readmissions)

In [ ]:
# Setup Silver Encounter Data
con.execute("""
CREATE TABLE encounter (
    person_id VARCHAR,
    encounter_id VARCHAR,
    encounter_start_date DATE,
    encounter_end_date DATE
)
""")

con.execute("""
INSERT INTO encounter VALUES
('P1', 'E1', '2023-01-01', '2023-01-05'),
('P1', 'E2', '2023-01-20', '2023-01-22'), -- 15 day readmission
('P2', 'E3', '2023-02-01', '2023-02-10'),
('P2', 'E4', '2023-04-01', '2023-04-05')  -- > 30 days, not a 30-day readmission
""")

# Simulate dbt logic for 30-day readmissions
readmissions_query = """
SELECT
    e1.person_id,
    e1.encounter_id AS index_encounter_id,
    e1.encounter_end_date AS index_discharge_date,
    e2.encounter_id AS readmission_encounter_id,
    e2.encounter_start_date AS readmission_date,
    date_diff('day', e1.encounter_end_date, e2.encounter_start_date) AS days_to_readmission
FROM encounter e1
JOIN encounter e2 
    ON e1.person_id = e2.person_id
    AND e1.encounter_id != e2.encounter_id
    AND e2.encounter_start_date > e1.encounter_end_date
    AND date_diff('day', e1.encounter_end_date, e2.encounter_start_date) <= 30
"""

print("Gold Mart: 30-Day Readmissions")
print(con.execute(readmissions_query).df())

## Simulate Dynamic Data Masking

In [ ]:
# Setup raw patient data
patients = pd.DataFrame({
    'person_id': ['P1', 'P2'],
    'ssn': ['123-45-6789', '987-65-4321'],
    'dob': ['1980-05-15', '1990-10-20']
})

def query_patients(role):
    df = patients.copy()
    if role != 'COMPLIANCE_OFFICER':
        # Apply Masking
        df['ssn'] = '***-**-****'
        df['dob'] = pd.to_datetime(df['dob']).dt.year.astype(str) + '-01-01' # Year start
    return df

print("Querying as DATA_SCIENTIST (Masked):")
print(query_patients('DATA_SCIENTIST'))

print("\nQuerying as COMPLIANCE_OFFICER (Unmasked):")
print(query_patients('COMPLIANCE_OFFICER'))